In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import networkx as nx
from itertools import permutations, combinations
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 600  # keep the same as dpi setting in figure.savefig
import matplotlib.lines as mlines
import matplotlib.dates as mdates

In [ ]:
#1. load study_area, regionalization and specify node locations on the map
study_area = gpd.read_file("/net/rain/hyclimm/data/projects/SynFire/Study_Area/Study_Area_32E.shp")    
bbox_adjusted = gpd.read_file("/net/rain/hyclimm/data/projects/SynFire/WP1/Regionalization/Region_Ten.shp") 
mctr_frac = pd.read_csv('/net/rain/hyclimm/data/projects/SynFire/WP1/multi_country_event_fraction/multi_country_event_fraction_2001_2020.csv')

region_positions = {}
region = ["BI", "IP", "FR", "ME", "AL", "SEA", "NEA", "SC", "WMD", "EMD"]  

region_positions['BI'] = (-5, 57)
region_positions['IP'] = (-5, 39.0)
region_positions['FR'] = (-2, 47)
region_positions['ME'] = (11.2, 53.3)
region_positions['AL'] = (11, 45.5)
region_positions['SEA'] = (23, 47)
region_positions['NEA'] = (28, 57)
region_positions['SC'] = (17, 68)
region_positions['WMD'] = (12, 36)
region_positions['EMD'] = (28, 39)

#-----------------------------------------
#2. node facecolor list, for degree centrality

node_colorlist = {
    
    '#03045e': 0,
    '#023e8a': 0.11,
    '#0077b6': 0.22,
    '#0096c7': 0.33,
    '#00b4d8': 0.44,
    '#48cae4': 0.55,
    '#90e0ef': 0.66,
    '#ade8f4': 0.77,
    '#caf0f8': 0.88,
    '#e3f2fd': 1
    
}

#-----------------------------------------
# 3. node edgecolor list, for fraction of cross-border events

node_edge_colorlist = ['#773CA6', #[0, 0.1)
                       '#A42095', #[0.1, 0.2)
                       '#CA3973', #[0.2, 0.3)
                       '#F66942', #[0.3, 0.4)
                       '#FEA545', #[0.4, 1]
                      ]

In [ ]:
#Figure main1
fig, axes = plt.subplots(2, 3, figsize = (19, 18))
axes = axes.flatten()

season_dict = dict(zip([0, 1, 2, 3, 4], ["all_seasons", "MAM", "JJA", "SON", "DJF"]))
title_dict = dict(zip([0, 1, 2, 3, 4], ["(a) All seasons", "(b) MAM", "(c) JJA", "(d) SON", "(e) DJF"]))
tau_max = 6
fire = "All_Fires"
scaling_factor = 1.2  # Adjust this factor for the thickness of the link

for ax_id, ax in enumerate(axes):

    #legend panel--------------------------------------------------------------------------------------------------
    if ax_id == 5:
        ax.axis("off")

        #-----------------------------------------------
        # Create custom legend handles with different colors for dots to denote the degree centrality
        dot_b0 = mlines.Line2D([], [], color='#03045e', marker='o', markersize=10, linestyle='None', label='0')
        dot_b1 = mlines.Line2D([], [], color='#023e8a', marker='o', markersize=10, linestyle='None', label='0.11')
        dot_b2 = mlines.Line2D([], [], color='#0077b6', marker='o', markersize=10, linestyle='None', label='0.22')
        dot_b3 = mlines.Line2D([], [], color='#0096c7', marker='o', markersize=10, linestyle='None', label='0.33')
        dot_b4 = mlines.Line2D([], [], color='#00b4d8', marker='o', markersize=10, linestyle='None', label='0.44')
        dot_b5 = mlines.Line2D([], [], color='#48cae4', marker='o', markersize=10, linestyle='None', label='0.55')
        dot_b6 = mlines.Line2D([], [], color='#90e0ef', marker='o', markersize=10, linestyle='None', label='0.66')
        dot_b7 = mlines.Line2D([], [], color='#ade8f4', marker='o', markersize=10, linestyle='None', label='0.77')
        dot_b8 = mlines.Line2D([], [], color='#caf0f8', marker='o', markersize=10, linestyle='None', label='0.88')
        dot_b9 = mlines.Line2D([], [], color='#e3f2fd', marker='o', markersize=10, linestyle='None', label='1')
        
        legend1 = ax.legend(handles=[dot_b0, dot_b5, dot_b1, dot_b6, dot_b2, dot_b7, dot_b3, dot_b8, dot_b4, dot_b9],
                               loc='upper left', fontsize=17, title='Degree centrality (-)', title_fontsize = 17, frameon = False, ncol = 5,  bbox_to_anchor=(0, 1.0), handletextpad = -0.5, columnspacing = 0)
        
        #-----------------------------------------------
        # Create custom legend handles with different colors for lines to denote the statistical significance of synchronicity
        red_line = mlines.Line2D([], [], color='#C80001', linewidth=2, label='Significant (p < 0.05)')
        dark_red_line = mlines.Line2D([], [], color='#7F0003', linewidth=2, label='Significant (p < 0.01)')
        grey_line = mlines.Line2D([], [], color='#909090', linewidth=2, label='Insignificant')
        
        legend2 = ax.legend(handles=[grey_line, red_line, dark_red_line],
                            loc='upper left', fontsize=17, title='Significance', title_fontsize = 17, frameon = False, bbox_to_anchor=(0.13, 0.82), handletextpad = 0.3)
        
        #-----------------------------------------------
        #create the legend for log(ES)
        ES_linewidth = [np.log(es) * scaling_factor for es in [5, 10, 25, 50, 100, 200, 400]]
        ES_label = [5, 10, 25, 50, 100, 200, 400]
    
        
        legend3 = ax.legend(handles=[mlines.Line2D([], [], color = '#909090', linewidth = lw, label = f"{lb:<4}",  # Pad labels to equal width
                                                     ) for lb, lw in zip(ES_label, ES_linewidth)],
                               loc = 'upper left', fontsize = 17, title = 'Event synchronicity (-)', title_fontsize = 17, frameon = False, bbox_to_anchor = (0.2, 0.6), ncol = 1, labelspacing = 0.3,  handletextpad = 0.3)
        
        #-----------------------------------------------
        # add legend for the fraction of cross-border events
        circ_b0 = mlines.Line2D([], [], color='w', marker='o', markerfacecolor = 'none', markeredgecolor = '#773CA6', markersize=32, markeredgewidth = 3.5, linestyle='None', label='[0, 0.1)')
        circ_b1 = mlines.Line2D([], [], color='w', marker='o', markerfacecolor = 'none', markeredgecolor = '#A42095', markersize=32, markeredgewidth = 3.5, linestyle='None', label='[0.1, 0.2)')
        circ_b2 = mlines.Line2D([], [], color='w', marker='o', markerfacecolor = 'none', markeredgecolor = '#CA3973', markersize=32, markeredgewidth = 3.5, linestyle='None', label='[0.2, 0.3)')
        circ_b3 = mlines.Line2D([], [], color='w', marker='o', markerfacecolor = 'none', markeredgecolor = '#F66942', markersize=32, markeredgewidth = 3.5, linestyle='None', label='[0.3, 0.4)')
        circ_b4 = mlines.Line2D([], [], color='w', marker='o', markerfacecolor = 'none', markeredgecolor = '#FEA545', markersize=32, markeredgewidth = 3.5, linestyle='None', label='[0.4, 1]')

        legend4 = ax.legend(handles=[circ_b0, circ_b3, circ_b1, circ_b4, circ_b2],
                            loc='upper left', fontsize=17, title='Fraction of cross-border events (-)', title_fontsize = 17, frameon = False, ncol = 3,  bbox_to_anchor=(0, 0.23), handletextpad = 0.2, labelspacing = 1.3, columnspacing = 0.5)
        
        
        # Add the first legend 
        ax.add_artist(legend1)
        ax.add_artist(legend2)
        ax.add_artist(legend3)

    #plot panel----------------------------------------------------------------------------------------------------
    else:
        
        #get season indicator
        season = season_dict[ax_id]

        display_ths = 10 if season == 'all_seasons' else 5

        
        #load event synchronicity matrix
        ES = pd.read_csv(f"/net/rain/hyclimm/data/projects/SynFire/WP1/Event_Sync_Sig_Sns/{fire}/Snstest/ES_matrix_taumax_{tau_max}_{season}.csv", index_col = 0)
    
        #aggregate the matrix to upper right
        for i in range(len(ES)):
            for j in range(i, len(ES)):
                if i == j:
                    ES.iloc[i,j] = 0
                else:
                    ES.iloc[i,j] += ES.iloc[j,i]
                    ES.iloc[j,i] = 0
                    
        #load significance test dataframe
        ES_sigtest = pd.read_csv(f"/net/rain/hyclimm/data/projects/SynFire/WP1/Event_Sync_Sig_Sns/{fire}/Sigtest/ES_Bpermutation_2000_taumax_{tau_max}_{season}.csv")
    
        #aggregate for each pairs
        for combo in combinations(region, 2):
            ES_sigtest[rf"{combo[0]}_{combo[1]}"] = ES_sigtest[rf"{combo[0]}_{combo[1]}"] + ES_sigtest[rf"{combo[1]}_{combo[0]}"]
            ES_sigtest[rf"{combo[1]}_{combo[0]}"] = np.nan
            
        ES_sigtest = ES_sigtest.dropna(axis = 1)
    
        #get significant pairs
        sig_pairs = []
        sig_pairs_p99 = []
        
        for combo in combinations(region, 2):
            if ES.loc[combo[0], combo[1]] > ES_sigtest.quantile(0.95)[f"{combo[0]}_{combo[1]}"]:
                sig_pairs.append((combo[0], combo[1]))
            if ES.loc[combo[0], combo[1]] > ES_sigtest.quantile(0.99)[f"{combo[0]}_{combo[1]}"]:
                sig_pairs_p99.append((combo[0], combo[1]))
                
        
        #--------------------------------------------------------------------------------------
        #construct networks for all links
        G = nx.Graph()
        
        #add nodes
        G.add_nodes_from(region)
        
        #add edges
        for region_combo in combinations(region, 2):
            syn = ES.loc[region_combo[0], region_combo[1]]
            G.add_edge(region_combo[0], region_combo[1], weight=syn)
                
        #edge weight
        weights = nx.get_edge_attributes(G, 'weight')
        scaled_weights = [np.log(weight) if weight >= display_ths else 0 for weight in weights.values()]
            
        edge_widths = [w * scaling_factor for w in scaled_weights]
        
        #--------------------------------------------------------------------------------------
        #construct a network with significant links only and calculate degree centrality
        G_sig = nx.Graph()

        
        #add nodes
        G_sig.add_nodes_from(region)

        
        #small changes to region abbreviations
        node_labels = {
        'BI':'BI',
        'IP':'IP',
        'FR':'FR',
        'ME':'CE', #ME->CE
        'AL':'AL',
        'SEA':'SEE',  #SEA->SEE
        'NEA':'NEE', #NEA->NEE
        'SC':'SC',
        'WMD':'WMD',
        'EMD':'EMD'
        }
        nx.set_node_attributes(G_sig, node_labels, "label")
        

        #add edges
        for sig_pair in sig_pairs:
            syn = ES.loc[sig_pair[0], sig_pair[1]]
            G_sig.add_edge(sig_pair[0], sig_pair[1], weight = syn, label = 'p99sig' if sig_pair in sig_pairs_p99 else 'p95sig')

        #edge weight
        weights_sig = nx.get_edge_attributes(G_sig, 'weight')
        scaled_weights_sig = [np.log(weight_sig) if weight_sig >= display_ths else 0 for weight_sig in weights_sig.values()]
    
        edge_widths_sig = [w * scaling_factor for w in scaled_weights_sig]

        #degree centrality
        degree_centrality = nx.degree_centrality(G_sig)
        degree_centrality = {node: round(centrality, 2) for node, centrality in degree_centrality.items()} #2 decimal

        #--------------------------------------------------------------------------------------
        #colors
        
        node_colors = []
        
        for node in G.nodes():
            node_colors.append([key for key, value in node_colorlist.items() if value == degree_centrality[node]][0])

        edge_colors = []

        for _, _, attrs in G_sig.edges(data = True):
            if attrs['label'] == 'p95sig':
                edge_colors.append('#C80001')
            if attrs['label'] == 'p99sig':
                edge_colors.append('#7F0003')
            
    
        #--------------------------------------------------------------------------------------
        #Draw the study area and regionalization
        study_area.boundary.plot(ax = ax, color = '#dddddd', zorder = 0)
        bbox_adjusted.boundary.plot(ax = ax, color='#ECECEC', zorder = 0)
        
        #Draw the network
        # Draw edges (grey for all links)
        nx.draw_networkx_edges(G, pos = region_positions, ax=ax, edge_color="#909090", width=edge_widths).set_zorder(1)

        #Draw the network for significant links
        nx.draw_networkx_edges(G_sig, pos = region_positions, ax=ax, edge_color = edge_colors, width=edge_widths_sig).set_zorder(2)
        
        # Draw nodes
        #node edge color
        f_multi = mctr_frac[mctr_frac['season'] == season].copy()
        f_multi_vals = [f_multi.loc[(f_multi['reg'] == r) & (f_multi['season'] == season)]['multi_country_fraction'].item() for r in region]
        color_ind = [0 if (np.isnan(f) or f < 0.1) 
                     else 1 if f < 0.2 
                     else 2 if f <0.3 
                     else 3 if f < 0.4 
                     else 4 
                     for f in f_multi_vals]
        node_edge_colors = [node_edge_colorlist[ind] for ind in color_ind]
        nx.draw_networkx_nodes(G_sig, pos = region_positions, ax=ax, node_color=node_colors, edgecolors = node_edge_colors, linewidths = 3.5, node_size=1300).set_zorder(3)
        
        # Draw labels
        nx.draw_networkx_labels(G_sig, pos = region_positions, ax=ax, labels = node_labels, font_color = "white", font_size=13, font_weight = "bold")
    
        # subplot label
        ax.text(0.04, 0.92, title_dict[ax_id], transform = ax.transAxes, fontsize = 20)
        
        # turn the axis on
        ax.set_axis_on()
        ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
        ax.set_xlabel("")
        ax.set_ylabel("")
        #--------------------------------------------------------------------------------------


plt.subplots_adjust(wspace = 0.04, hspace = 0.02) 
fig.savefig(f"/home/lixinh/SynFire_Scripts/WP1/Figures/v5_1_SynMap.png", dpi = 600, bbox_inches = "tight")
plt.show()

In [ ]:
#Figure S1-S5
def plot_fire_synchronicity_sensitivity_map(season):

    display_ths = 10 if season == 'all_seasons' else 5
    
    fig, axes = plt.subplots(2, 3, figsize = (19, 18))
    axes = axes.flatten()
    
    
    #tau_max values
    taumax_list = [9, 12, 15, 18, 21]

    #subpanel labels
    sub_title_dict = ["(a) $\\tau_{max}$ = 9 days", "(b) $\\tau_{max}$ = 12 days", 
                      "(c) $\\tau_{max}$ = 15 days", "(d) $\\tau_{max}$ = 18 days", "(e) $\\tau_{max}$ = 21 days"]
    
    fire = "All_Fires"
    scaling_factor = 1.2  # Adjust this factor for the thickness of the link
    
    for ax_id, ax in enumerate(axes):
    
        #legend panel--------------------------------------------------------------------------------------------------
        if ax_id == 5:
            ax.axis("off")
            
            # Create custom legend handles with different colors for dots to denote the degree centrality
            dot_b0 = mlines.Line2D([], [], color='#03045e', marker='o', markersize=10, linestyle='None', label='0')
            dot_b1 = mlines.Line2D([], [], color='#023e8a', marker='o', markersize=10, linestyle='None', label='0.11')
            dot_b2 = mlines.Line2D([], [], color='#0077b6', marker='o', markersize=10, linestyle='None', label='0.22')
            dot_b3 = mlines.Line2D([], [], color='#0096c7', marker='o', markersize=10, linestyle='None', label='0.33')
            dot_b4 = mlines.Line2D([], [], color='#00b4d8', marker='o', markersize=10, linestyle='None', label='0.44')
            dot_b5 = mlines.Line2D([], [], color='#48cae4', marker='o', markersize=10, linestyle='None', label='0.55')
            dot_b6 = mlines.Line2D([], [], color='#90e0ef', marker='o', markersize=10, linestyle='None', label='0.66')
            dot_b7 = mlines.Line2D([], [], color='#ade8f4', marker='o', markersize=10, linestyle='None', label='0.77')
            dot_b8 = mlines.Line2D([], [], color='#caf0f8', marker='o', markersize=10, linestyle='None', label='0.88')
            dot_b9 = mlines.Line2D([], [], color='#e3f2fd', marker='o', markersize=10, linestyle='None', label='1')
            
            legend1 = ax.legend(handles=[dot_b0, dot_b5, dot_b1, dot_b6, dot_b2, dot_b7, dot_b3, dot_b8, dot_b4, dot_b9],
                                   loc='upper left', fontsize=17, title='Degree centrality (-)', title_fontsize = 17, frameon = False, ncol = 5,  bbox_to_anchor=(0, 1.0), handletextpad = -0.5, columnspacing = 0)
            
            # Create custom legend handles with different colors for lines to denote the statistical significance of synchronicity
            red_line = mlines.Line2D([], [], color='#d11d27', linewidth=2, label='Significant (p < 0.05)')
            grey_line = mlines.Line2D([], [], color='#909090', linewidth=2, label='Insignificant')
            
            legend2 = ax.legend(handles=[grey_line, red_line],
                                loc='upper left', fontsize=17, title='Significance', title_fontsize = 17, frameon = False, bbox_to_anchor=(0.13, 0.73), handletextpad = 0.3)
    
            #create the legend for log(ES)
            ES_linewidth = [np.log(es) * scaling_factor for es in [5, 10, 25, 50, 100, 200, 400]]
            ES_label = [5, 10, 25, 50, 100, 200, 400]
        
            
            legend3 = ax.legend(handles=[mlines.Line2D([], [], color = '#909090', linewidth = lw, label = f"{lb:<4}",  # Pad labels to equal width
                                                         ) for lb, lw in zip(ES_label, ES_linewidth)],
                                   loc = 'upper left', fontsize = 17, title = 'Event synchronicity (-)', title_fontsize = 17, frameon = False, bbox_to_anchor = (0.2, 0.45), ncol = 1, handletextpad = 0.3)
            
            # Add the first legend 
            ax.add_artist(legend1)
            ax.add_artist(legend2)
    
        #plot panel----------------------------------------------------------------------------------------------------
        else:
            
            #get tau_max
            tau_max = taumax_list[ax_id]
    
            
            #load event synchronicity matrix
            ES = pd.read_csv(f"/net/rain/hyclimm/data/projects/SynFire/WP1/Event_Sync_Sig_Sns/{fire}/Snstest/ES_matrix_taumax_{tau_max}_{season}.csv", index_col = 0)
        
            #aggregate the matrix to upper right
            for i in range(len(ES)):
                for j in range(i, len(ES)):
                    if i == j:
                        ES.iloc[i,j] = 0
                    else:
                        ES.iloc[i,j] += ES.iloc[j,i]
                        ES.iloc[j,i] = 0
                        
            #load significance test dataframe
            ES_sigtest = pd.read_csv(f"/net/rain/hyclimm/data/projects/SynFire/WP1/Event_Sync_Sig_Sns/{fire}/Sigtest/ES_Bpermutation_2000_taumax_{tau_max}_{season}.csv")
        
            #aggregate for each pairs
            for combo in combinations(region, 2):
                ES_sigtest[rf"{combo[0]}_{combo[1]}"] = ES_sigtest[rf"{combo[0]}_{combo[1]}"] + ES_sigtest[rf"{combo[1]}_{combo[0]}"]
                ES_sigtest[rf"{combo[1]}_{combo[0]}"] = np.nan
                
            ES_sigtest = ES_sigtest.dropna(axis = 1)
        
            #get significant pairs
            sig_pairs = []
            
            for combo in combinations(region, 2):
                if ES.loc[combo[0], combo[1]] > ES_sigtest.quantile(0.95)[f"{combo[0]}_{combo[1]}"]:
                    sig_pairs.append((combo[0], combo[1]))
                    
            
            #--------------------------------------------------------------------------------------
            #construct networks for all links
            G = nx.Graph()
            
            #add nodes
            G.add_nodes_from(region)
            
            #add edges
            for region_combo in combinations(region, 2):
                syn = ES.loc[region_combo[0], region_combo[1]]
                G.add_edge(region_combo[0], region_combo[1], weight=syn)
                    
            #edge weight
            weights = nx.get_edge_attributes(G, 'weight')
            scaled_weights = [np.log(weight) if weight >= display_ths else 0 for weight in weights.values()]
                
            edge_widths = [w * scaling_factor for w in scaled_weights]
            
            #--------------------------------------------------------------------------------------
            #construct a network with significant links only and calculate degree centrality
            G_sig = nx.Graph()
    
            #add nodes
            G_sig.add_nodes_from(region)

            #small changes to region abbreviations
            node_labels = {
            'BI':'BI',
            'IP':'IP',
            'FR':'FR',
            'ME':'CE', #ME->CE
            'AL':'AL',
            'SEA':'SEE',  #SEA->SEE
            'NEA':'NEE', #NEA->NEE
            'SC':'SC',
            'WMD':'WMD',
            'EMD':'EMD'
            }
            nx.set_node_attributes(G_sig, node_labels, "label")
    
    
            #add edges
            for sig_pair in sig_pairs:
                syn = ES.loc[sig_pair[0], sig_pair[1]]
                G_sig.add_edge(sig_pair[0], sig_pair[1], weight = syn)
    
            #edge weight
            weights_sig = nx.get_edge_attributes(G_sig, 'weight')
            scaled_weights_sig = [np.log(weight_sig) if weight_sig >= display_ths else 0 for weight_sig in weights_sig.values()]
        
            edge_widths_sig = [w * scaling_factor for w in scaled_weights_sig]
    
            #degree centrality
            degree_centrality = nx.degree_centrality(G_sig)
            degree_centrality = {node: round(centrality, 2) for node, centrality in degree_centrality.items()} #2 decimal
        
            node_colors = []
            
            for node in G.nodes():
                node_colors.append([key for key, value in node_colorlist.items() if value == degree_centrality[node]][0])
        
            #--------------------------------------------------------------------------------------
            #Draw the study area and regionalization
            study_area.boundary.plot(ax = ax, color = '#dddddd', zorder = 0)
            bbox_adjusted.boundary.plot(ax = ax, color='#ECECEC', zorder = 0)
            
            #Draw the network
            # Draw edges (grey for all links)
            nx.draw_networkx_edges(G, pos = region_positions, ax=ax, edge_color="#909090", width=edge_widths).set_zorder(1)
    
            #Draw the network for significant links
            nx.draw_networkx_edges(G_sig, pos = region_positions, ax=ax, edge_color = "#d11d27", width=edge_widths_sig).set_zorder(2)
            
            # Draw nodes
            nx.draw_networkx_nodes(G_sig, pos = region_positions, ax=ax, node_color=node_colors, node_size=1300).set_zorder(3)
            
            # Draw labels
            nx.draw_networkx_labels(G_sig, pos = region_positions, ax=ax, labels = node_labels, font_color = "white", font_size=13, font_weight = "bold")
        
            # subplot label
            ax.text(0.04, 0.92, sub_title_dict[ax_id], transform = ax.transAxes, fontsize = 20)
            
            # turn the axis on
            ax.set_axis_on()
            ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
            ax.set_xlabel("")
            ax.set_ylabel("")
            #--------------------------------------------------------------------------------------
    
    
    plt.subplots_adjust(wspace = 0.04, hspace = 0.02) 
    fig.savefig(f"/home/lixinh/SynFire_Scripts/WP1/Figures/R_CEE_Sensitivity_Maps_{season}.png", dpi = 600, bbox_inches = "tight")
    plt.show()

In [ ]:
for s in ["MAM", "JJA", "SON", "DJF", "all_seasons"]:
    plot_fire_synchronicity_sensitivity_map(s)